In [ ]:
import argparse
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import torchmetrics as tm
import albumentations as A
from collections import OrderedDict

import sys 
sys.path.append('main-codes/Mamba-UNet-main/code')
from networks.vision_transformer import SwinUnet as ViT_seg
from networks.vision_mamba import MambaUnet as VIM_seg

sys.path.append("main-codes/modified_segmentation_models-main") 
import segmentation_models_pytorch as smp 

sys.path.append("main-codes/HybridModels")
from HybridSwinUNetC import HybridSwinUNetC
from HybridMambaUNetC import HybridMambaUNetC

from config import get_config

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
torch.cuda.empty_cache()

In [ ]:
class MultiClassSegDataset(Dataset):
    
    def __init__(self, df, transform=None,):
        self.df = df
        self.transform = transform
    
    def __getitem__(self, idx):
        image_name = self.df.iloc[idx, 1]
        mask_name = self.df.iloc[idx, 2]
        image = cv2.imread(image_name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_name, cv2.IMREAD_UNCHANGED)
        image = image.astype('uint8')
        mask = mask[:,:,0]
        if(self.transform is not None):
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)
        image = image.permute(2, 0, 1)
        image = image.float()/255
        mask = mask.long()
        return image, mask  
        
    def __len__(self):
        return len(self.df)

In [ ]:
INTPUT_DIR = "" # Path to the landcover.ai dataset
CLASSES = ['background', 'building', 'woodlands', 'water', 'road']

testSet = pd.read_csv(INTPUT_DIR + "test.txt", header=None, names=["file"])
testSet["img"] = INTPUT_DIR + "output/" + testSet['file'] + ".jpg"
testSet["mask"] = INTPUT_DIR + "output/" + testSet['file'] + "_m.png"

print("Testing dataset: " + str(len(testSet)))

test_transform = A.Compose(
    [A.CenterCrop(height=224, width=224),])

testDF = testSet.sample(n=1600, random_state=42)

testDS = MultiClassSegDataset(testDF, transform=test_transform)

testDL = DataLoader(testDS, batch_size=8, shuffle=False,
                    num_workers=0, pin_memory=False, drop_last=False)

print("Testing dataset: " + str(len(testDS)))

In [ ]:
# Argument Parser for Mamba-based models
mamba_parser = argparse.ArgumentParser()
mamba_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
mamba_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised', help='experiment_name')
mamba_parser.add_argument('--model', type=str, default='mambaunet', help='model_name')
mamba_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')

mamba_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml", help='path to config file')
mamba_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs.")

mamba_parser.add_argument('--zip', action='store_true', help='use zipped dataset instead of folder dataset')
mamba_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'])
mamba_parser.add_argument('--resume', help='resume from checkpoint')
mamba_parser.add_argument('--accumulation-steps', type=int)
mamba_parser.add_argument('--use-checkpoint', action='store_true')
mamba_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'])
mamba_parser.add_argument('--tag')
mamba_parser.add_argument('--eval', action='store_true')
mamba_parser.add_argument('--throughput', action='store_true')

mamba_parser.add_argument('--max_iterations', type=int, default=25)
mamba_parser.add_argument('--batch_size', type=int, default=8)
mamba_parser.add_argument('--deterministic', type=int, default=1)
mamba_parser.add_argument('--base_lr', type=float, default=0.0001)
mamba_parser.add_argument('--patch_size', type=int, nargs=2, default=[224, 224])
mamba_parser.add_argument('--seed', type=int, default=1337)
mamba_parser.add_argument('--labeled_num', type=int, default=140)

# Step 2: Simulated args for Jupyter/IPython
mamba_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised',
    '--model', 'mambaunet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml',
    '--patch_size', '224', '224',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
mamba_args = mamba_parser.parse_args(args=mamba_simulated_args)

# Step 3: Load config
mamba_config = get_config(mamba_args)

In [ ]:
# Argument Parser for Swin-based models
swin_parser = argparse.ArgumentParser()
swin_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
swin_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised_ViT', help='experiment_name')
swin_parser.add_argument('--model', type=str, default='unet', help='model_name')
swin_parser.add_argument('--max_iterations', type=int, default=25, help='maximum epoch number to train')
swin_parser.add_argument('--batch_size', type=int, default=8, help='batch_size per gpu')
swin_parser.add_argument('--deterministic', type=int, default=1, help='whether use deterministic training')
swin_parser.add_argument('--base_lr', type=float, default=0.0001, help='segmentation network learning rate')
swin_parser.add_argument('--patch_size', type=list, default=[224, 224], help='patch size of network input')
swin_parser.add_argument('--seed', type=int,  default=1337, help='random seed')
swin_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')
swin_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml", help='path to config file', )
swin_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs. ")
swin_parser.add_argument('--zip', action='store_true',
                    help='use zipped dataset instead of folder dataset')
swin_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'],
                    help='no: no cache, '
                         'full: cache all data, '
                         'part: sharding the dataset into nonoverlapping pieces and only cache one piece')
swin_parser.add_argument('--resume', help='resume from checkpoint')
swin_parser.add_argument('--accumulation-steps', type=int,
                    help="gradient accumulation steps")
swin_parser.add_argument('--use-checkpoint', action='store_true',
                    help="whether to use gradient checkpointing to save memory")
swin_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'],
                    help='mixed precision opt level, if O0, no amp is used')
swin_parser.add_argument('--tag', help='tag of experiment')
swin_parser.add_argument('--eval', action='store_true',
                    help='Perform evaluation only')
swin_parser.add_argument('--throughput', action='store_true',
                    help='Test throughput only')

# label and unlabel
swin_parser.add_argument('--labeled_bs', type=int, default=12,
                    help='labeled_batch_size per gpu')
swin_parser.add_argument('--labeled_num', type=int, default=140,
                    help='labeled data')
# costs
swin_parser.add_argument('--ema_decay', type=float,  default=0.99, help='ema_decay')
swin_parser.add_argument('--consistency_type', type=str,
                    default="mse", help='consistency_type')
swin_parser.add_argument('--consistency', type=float,
                    default=0.1, help='consistency')
swin_parser.add_argument('--consistency_rampup', type=float,
                    default=200.0, help='consistency_rampup')

# Step 2: Simulated args for Jupyter/IPython
swin_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised_ViT',
    '--model', 'unet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml',
    '--patch_size', '[224, 224]',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
swin_args = swin_parser.parse_args(args=swin_simulated_args)

# Step 3: Load config
swin_config = get_config(swin_args)

In [ ]:
acc = tm.Accuracy(task="multiclass", average="micro", num_classes=5).to(device)
f1 = tm.F1Score(task="multiclass", average="macro", num_classes=5).to(device)
recall = tm.Recall(task="multiclass", average="macro", num_classes=5).to(device)
precision = tm.Precision(task="multiclass", average="macro", num_classes=5).to(device)

In [ ]:
met_df = pd.DataFrame(columns=['Model', 'SetNum', 'Accuracy', 'F1_score', 'Recall', 'Precision'])

In [ ]:
def AssessPipe(model_name, setNum, chckpnt):

    if model_name == "SwinUNet":
        model = ViT_seg(swin_config, 
                img_size=swin_args.patch_size,
                num_classes=swin_args.num_classes).to(device)
    elif model_name == "SwinUNetC":
        model = HybridSwinUNetC(swin_config, 
            img_size=swin_args.patch_size,
            num_classes=swin_args.num_classes).to(device)
    elif model_name == "MambaUNet":
        model = VIM_seg(mamba_config, 
                img_size=mamba_args.patch_size,
                num_classes=mamba_args.num_classes).to(device)
    elif model_name == "MambaUNetC":
        model = HybridMambaUNetC(mamba_config, 
                                    img_size=mamba_args.patch_size, 
                                    num_classes=mamba_args.num_classes).to(device) 
    else:
        model = smp.Unet(
            encoder_name = "resnet34",
            encoder_weights = "imagenet",
            decoder_channels = (768, 768, 384, 192, 96),
            in_channels = 3,
            classes = 5
            ).to(device)

    print("The model is: ", model_name)

    # Load the state dict
    checkpoint = torch.load(chckpnt)

    # Remove `module.` prefix from keys
    new_state_dict = OrderedDict()
    for k, v in checkpoint.items():
        new_key = k.replace("module.", "")  # Remove "module." from the key
        new_state_dict[new_key] = v
    print(model.load_state_dict(new_state_dict))
    metric = []
    model.eval()
    with torch.no_grad():
            for batch_idx, (inputsTe, targetsTe) in enumerate(testDL):
                # Get data and move to device
                inputsTe, targetsTe = inputsTe.to(device), targetsTe.to(device)
                # Predict data
                outputsTe = model(inputsTe)
                # Calculate metrics
                accTe = acc(outputsTe, targetsTe)
                f1Te = f1(outputsTe, targetsTe)
                recallTe = recall(outputsTe, targetsTe)
                precisionTe = precision(outputsTe, targetsTe)

    accTe = acc.compute()
    f1Te = f1.compute()
    recallTe = recall.compute()
    precisionTe = precision.compute()
    
    metric.append(accTe.detach().cpu().numpy())
    metric.append(f1Te.detach().cpu().numpy())
    metric.append(recallTe.detach().cpu().numpy())
    metric.append(precisionTe.detach().cpu().numpy())

    acc.reset()
    f1.reset()
    recall.reset()
    precision.reset()

    global met_df
    new_row = pd.Series({
        'Model' : model_name,
        'SetNum' : setNum,
        'Accuracy': metric[0], 
        'F1_score': metric[1], 
        'Recall': metric[2],
        'Precision': metric[3]})
    met_df = pd.concat([met_df, 
                        new_row.to_frame().T], 
                        ignore_index=True)
    print(met_df)

    torch.cuda.empty_cache()

In [ ]:
MODELS_DIR = "" # Path to saved models
set = ["S1", "S2", "S3"]
arch = ["SwinUNet", "SwinUNetC", "MambaUNet", "MambaUNetC","UNet"]

for a in arch:
    for s in set:
        AssessPipe(model_name = a, 
                   setNum = s, 
                   chckpnt = MODELS_DIR + a + "_" + s + ".pt")

In [ ]:
RESULT_DIR = "" # Path to save metrics
met_df.to_csv(RESULT_DIR + "Testing_mets.csv")